# 初始化

欢迎来到《改进深度神经网络》的第一篇作业！

训练神经网络需要指定权重的初始值。精心选择的初始化方法有助于学习过程。

如果你完成了本专项课程的上一课程，你可能遵循了权重初始化的说明，并且发现到目前为止效果非常好。但是如何为新的神经网络选择初始化呢？在此 Notebook 中，你将尝试几种不同的初始化，包括随机、零和 He 初始化，并了解每种初始化如何导致不同的结果。

精心选择的初始化可以：
- 加快梯度下降的收敛速度
- 增加梯度下降收敛到较低训练（和泛化）误差的几率 

让我们开始吧！

## 关于提交至 AutoGrader 的重要说明

向 AutoGrader 提交作业前，请确认满足以下要求：

1. 未在作业中添加 _额外的_  `print` 语句。
2. 未在作业中添加 _额外的_ 代码单元格。
3. 未修改任何函数的参数。
4. 未在计分练习中使用全局变量。除非明确要求使用全局变量，否则应使用局部变量。
5. 未修改无需改动的作业代码，例如创建 _额外的_ 变量。

如果违反上述要求，提交作业时可能出现 `Grader Error: Grader feedback not found` 或其他意外错误。在求助或调试作业之前，请先检查这些情况。如果不记得自己做过哪些改动，可以按照 [说明](https://www.coursera.org/learn/deep-neural-network/supplement/QWEnZ/h-ow-to-refresh-your-workspace) 获取一份新的作业。

## 目录
- [1 - 依赖库](#1)
- [2 - 加载数据集](#2)
- [3 - 神经网络模型](#3)
- [4 - 零初始化](#4)
    - [练习 1 - initialize_parameters_zeros](#ex-1)
- [5 - 随机初始化](#5)
    - [练习 2 - initialize_parameters_random](#ex-2)
- [6 - He 初始化](#6)
    - [练习 3 - initialize_parameters_he](#ex-3)
- [7 - 结论](#7)

<a name='1'></a>
## 1 - 依赖库

In [ ]:
### v1.1

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sklearn
import sklearn.datasets
from public_tests import *
from init_utils import sigmoid, relu, compute_loss, forward_propagation, backward_propagation
from init_utils import update_parameters, predict, load_dataset, plot_decision_boundary, predict_dec

%matplotlib inline
plt.rcParams['figure.figsize'] = (7.0, 4.0) # 设置绘图的默认大小
plt.rcParams['image.interpolation'] = 'nearest'
plt.rcParams['image.cmap'] = 'gray'

%load_ext autoreload
%autoreload 2

# 加载图像数据集：圆圈中的蓝色/红色点
# train_X, train_Y, test_X, test_Y = load_dataset()

<a name='2'></a>
## 2 - 加载数据集

In [ ]:
train_X, train_Y, test_X, test_Y = load_dataset()

对于此分类器，你想要将蓝点与红点分开。

<a name='3'></a>
## 3 - 神经网络模型 

你将使用 3 层神经网络（已为你实现）。这些是你将尝试的初始化方法： 
- *零初始化* -- 在输入参数中设置 `initialization = "zeros"`。
- *随机初始化* -- 在输入参数中设置 `initialization = "random"`。这会将权重初始化为大的随机值。  
- *他初始化* -- 在输入参数中设置 `initialization = "he"`。这会将权重初始化为根据 He 等人 2015 年发表的论文缩放的随机值。 

**说明**： 说明：阅读下面的代码，然后运行它。在下一部分中，你将实现 `model()` 调用的三个初始化方法。

In [ ]:
def model(X, Y, learning_rate = 0.01, num_iterations = 15000, print_cost = True, initialization = "he"):
    """
    实现三层神经网络：LINEAR->ReLU->LINEAR->ReLU->LINEAR->sigmoid。
    
    参数：
    X -- 输入数据，形状（2，示例数量）
    Y -- 真实的 "label" 向量（包含 0 代表红点；1 代表蓝点），形状（1，示例数量）
    learning_rate -- 学习率代表梯度下降 
    num_iterations -- 运行的迭代次数梯度下降
    print_cost -- 如果为 True，则每 1000 次迭代打印代价
    initialization -- 选择使用哪个初始化的标志（"zeros"、"random" 或 "he"）
    
    返回：
    parameters -- 模型学习到的参数
    """
        
    grads = {}
    costs = [] # 跟踪损失
    m = X.shape[1] # 示例数量
    layers_dims = [X.shape[0], 10, 5, 1]
    
    # 初始化参数字典。
    if initialization == "zeros":
        parameters = initialize_parameters_zeros(layers_dims)
    elif initialization == "random":
        parameters = initialize_parameters_random(layers_dims)
    elif initialization == "he":
        parameters = initialize_parameters_he(layers_dims)

    # 循环 (梯度下降)

    for i in range(num_iterations):

        # 前向传播：线性 -> ReLU -> 线性 -> ReLU -> 线性 -> sigmoid。
        a3, cache = forward_propagation(X, parameters)
        
        # 损失
        cost = compute_loss(a3, Y)

        # 反向传播.
        grads = backward_propagation(X, Y, cache)
        
        # 更新参数。
        parameters = update_parameters(parameters, grads, learning_rate)
        
        # 每 1000 次迭代打印损失
        if print_cost and i % 1000 == 0:
            print("Cost after iteration {}: {}".format(i, cost))
            costs.append(cost)
            
    # 绘制损失图
    plt.plot(costs)
    plt.ylabel('cost')
    plt.xlabel('iterations (per hundreds)')
    plt.title("Learning rate =" + str(learning_rate))
    plt.show()
    
    return parameters

<a name='4'></a>
## 4 - 零初始化

神经网络中有两种类型的参数需要初始化：
- 权重矩阵 $(W^{[1]}, W^{[2]}, W^{[3]}, ..., W^{[L-1]}, W^{[L]})$
- 偏置向量 $(b^{[1]}, b^{[2]}, b^{[3]}, ..., b^{[L-1]}, b^{[L]})$

<a name='ex-1'></a>
### 练习 1 - initialize_parameters_zeros

实现以下函数将所有参数初始化为零。稍后你会发现这效果不佳，因为它无法“破坏对称性”，但无论如何都要尝试一下，看看会发生什么。使用具有正确形状的 `np.zeros((..,..))`。

In [ ]:
# GRADED FUNCTION: initialize_parameters_zeros 

def initialize_parameters_zeros(layers_dims):
    """
    参数：
    layer_dims -- 包含每层大小的 python 数组（列表）。
    
    返回：
    parameters -- 包含参数 "W1"、"b1"、...、"WL"、"bL" 的 python 字典：
                    W1 -- 形状权重矩阵(layers_dims[1], layers_dims[0])
                    b1 -- 形状的偏置向量 (layers_dims[1], 1)
                    ...
                    WL -- 形状权重矩阵(layers_dims[L], layers_dims[L-1])
                    bL -- 形状的偏置向量 (layers_dims[L], 1)
    """
    
    parameters = {}
    L = len(layers_dims)            # 网络层数
    
    for l in range(1, L):
        #（约2行代码）
        # parameters['W' + str(l)] = 
        # parameters['b' + str(l)] = 
        # YOUR CODE STARTS HERE
        
        
        # YOUR CODE ENDS HERE
    return parameters

In [ ]:
parameters = initialize_parameters_zeros([3, 2, 1])
print("W1 = " + str(parameters["W1"]))
print("b1 = " + str(parameters["b1"]))
print("W2 = " + str(parameters["W2"]))
print("b2 = " + str(parameters["b2"]))
initialize_parameters_zeros_test(initialize_parameters_zeros)


运行以下代码，使用零初始化来训练你的模型 15,000 次迭代。

In [ ]:
parameters = model(train_X, train_Y, initialization = "zeros")
print ("On the train set:")
predictions_train = predict(train_X, train_Y, parameters)
print ("On the test set:")
predictions_test = predict(test_X, test_Y, parameters)

性能很糟糕，代价没有降低，算法的性能并不比随机猜测好。为什么？看一下预测和决策边界的详细信息：

In [ ]:
print ("predictions_train = " + str(predictions_train))
print ("predictions_test = " + str(predictions_test))

In [ ]:
plt.title("Model with Zeros initialization")
axes = plt.gca()
axes.set_xlim([-1.5,1.5])
axes.set_ylim([-1.5,1.5])
plot_decision_boundary(lambda x: predict_dec(parameters, x.T), train_X, train_Y)

要获得对此的全面解释，你可以阅读 *Paul Mielke* 的帖子 [对称破缺与零初始化](https://community.deeplearning.ai/t/symmetry-breaking-versus-zero-initialization/16061)。

简单解释如下：

__注意__：为了简单起见，下面的计算一次仅使用一个示例。

由于权重和偏置为零，因此乘以权重会创建零向量，当激活函数为 ReLU 时，该零向量给出 0。如`z = 0`

$$a = ReLU(z) = max(0, z) = 0$$

在分类层，其中激活函数是 sigmoid，然后你将得到（对于任一输入）： 

$$\sigma(z) = \frac{1}{ 1 + e^{-(z)}} = \frac{1}{2} = y_{pred}$$

对于每个例子，你有 0.5 的机会它是真的，我们的代价函数在调整权重时变得无能为力。

你的损失函数：
$$ \mathcal{L}(a, y) =  - y  \ln(y_{pred}) - (1-y)  \ln(1-y_{pred})$$

对于 `y=1`，`y_pred=0.5` 变为：

$$ \mathcal{L}(0, 1) =  - (1)  \ln(\frac{1}{2}) = 0.6931471805599453$$

对于 `y=0`，`y_pred=0.5` 变为：

$$ \mathcal{L}(0, 0) =  - (1)  \ln(\frac{1}{2}) = 0.6931471805599453$$

正如你所看到的，预测值为 0.5，无论实际 (`y`) 值是 1 还是 0，你都会得到相同的损失值，因此没有任何权重得到调整，并且你将陷入相同的旧权重值。 

这就是为什么你可以看到模型对每个示例都预测 0！难怪它的表现如此糟糕。

一般来说，将所有权重初始化为零会导致网络无法打破对称性。这意味着每一层中的每个神经元都会学习相同的东西，因此你不妨为每一层训练一个使用 $n^{[l]}=1$ 的神经网络。这样，网络并不比逻辑回归等线性分类器更强大。 

<div style="color:blue;">

**你应该记住什么**：
- 权重 $W^{[l]}$ 应随机初始化以破坏对称性。 
- 但是，将偏置 $b^{[l]}$ 初始化为零是可以的。只要 $W^{[l]}$ 是随机初始化的，对称性仍然会被破坏。

</div>

<a name='5'></a>
## 5 - 随机初始化

要打破对称性，请随机初始化权重。在随机初始化之后，每个神经元可以继续学习其输入的不同函数。在本练习中，你将看到当权重随机初始化但初始化为非常大的值时会发生什么。

<a name='ex-2'></a>
### 练习 2 - initialize_parameters_random

实现以下函数将权重初始化为大随机值（按 \*10 缩放）并将偏置初始化为零。使用 `np.random.randn(..,..) * 10` 作为权重，使用 `np.zeros((.., ..))` 作为偏置。你使用固定的 `np.random.seed(..)` 来确保你的“随机”权重与我们的相匹配，因此如果多次运行代码总是为你提供相同的参数初始值，请不要担心。 

In [ ]:
# GRADED FUNCTION: initialize_parameters_random

def initialize_parameters_random(layers_dims):
    """
    参数：
    layer_dims -- 包含每层大小的 python 数组（列表）。
    
    返回：
    parameters -- 包含参数 "W1"、"b1"、...、"WL"、"bL" 的 python 字典：
                    W1 -- 形状权重矩阵(layers_dims[1], layers_dims[0])
                    b1 -- 形状的偏置向量 (layers_dims[1], 1)
                    ...
                    WL -- 形状权重矩阵(layers_dims[L], layers_dims[L-1])
                    bL -- 形状的偏置向量 (layers_dims[L], 1)
    """
    
    np.random.seed(3)               # 这个种子确保你的 "random" 号码将与我们的一样
    parameters = {}
    L = len(layers_dims)            # 表示层数的整数
    
    for l in range(1, L):
        #（约2行代码）
        # parameters['W' + str(l)] = 
        # parameters['b' + str(l)] =
        # YOUR CODE STARTS HERE
        
        
        # YOUR CODE ENDS HERE

    return parameters

In [ ]:
parameters = initialize_parameters_random([3, 2, 1])
print("W1 = " + str(parameters["W1"]))
print("b1 = " + str(parameters["b1"]))
print("W2 = " + str(parameters["W2"]))
print("b2 = " + str(parameters["b2"]))
initialize_parameters_random_test(initialize_parameters_random)

运行以下代码，使用随机初始化对模型进行 15,000 次迭代训练。

In [ ]:
parameters = model(train_X, train_Y, initialization = "random")
print ("On the train set:")
predictions_train = predict(train_X, train_Y, parameters)
print ("On the test set:")
predictions_test = predict(test_X, test_Y, parameters)

如果你看到“inf”作为迭代 0 后的代价，这是因为数值舍入。一个更复杂的数值实现可以解决这个问题，但就 Notebook 的目的而言，这并不值得担心。

无论如何，你现在已经打破了对称性，这比以前提供了明显更好的准确性。该模型不再输出全 0。进步！

In [ ]:
print (predictions_train)
print (predictions_test)

In [ ]:
plt.title("Model with large random initialization")
axes = plt.gca()
axes.set_xlim([-1.5,1.5])
axes.set_ylim([-1.5,1.5])
plot_decision_boundary(lambda x: predict_dec(parameters, x.T), train_X, train_Y)

**观察**：
- 代价非常高。这是因为对于较大的随机值权重，最后一个激活值 (sigmoid) 输出的结果在某些示例中非常接近 0 或 1，并且当该示例错误时，该示例会产生非常高的损失。事实上，当 $\log(a^{[3]}) = \log(0)$ 时，损失变为无穷大。
- 初始化不当会导致消失/梯度爆炸，这也会减慢优化算法的速度。 
- 如果你训练这个网络的时间更长，你会看到更好的结果，但是用太大的随机数初始化会减慢优化速度。

<div style="color:blue;">

**总结**：
- 将权重初始化为非常大的随机值效果不佳。 
- 使用小的随机值初始化应该会更好。重要的问题是，这些随机值应该有多小？接下来我们就来了解一下吧！

</div>    
    
**可选阅读：**


高斯变量（`numpy.random.randn()`）和均匀随机变量的主要区别在于生成的随机数的分布：

- numpy.random.rand() 生成 [均匀分布](https://raw.githubusercontent.com/jahnog/deeplearning-notes/master/Course2/images/rand.jpg) 中的数字。
- 和 numpy.random.randn() 生成 [正态分布](https://raw.githubusercontent.com/jahnog/deeplearning-notes/master/Course2/images/randn.jpg) 中的数字。

当用于权重初始化时，randn() 帮助大多数权重避免接近极端，将大部分权重分配在范围的中心。

直观的查看方法是，例如，如果你使用 [sigmoid() 激活函数](https://raw.githubusercontent.com/jahnog/deeplearning-notes/master/Course2/images/sigmoid.jpg)。

你会记得，接近 0 或接近 1 的斜率非常小，因此这些极值附近的权重收敛到解的速度要慢得多，而将它们中的大多数放在中心附近会加快收敛速度。

<a name='6'></a>
## 6 - He 初始化

最后尝试“He初始化”；这是以 He 等人，2015 年的第一作者命名的。（如果你听说过“Xavier 初始化”，那么这很相似，除了 Xavier 初始化使用 `sqrt(1./layers_dims[l-1])` 的权重 $W^{[l]}$ 的缩放因子，其中 He 初始化将使用 `sqrt(2./layers_dims[l-1])`。）

<a name='ex-3'></a>
### 练习 3 - initialize_parameters_he

实现以下函数以使用 He 初始化来初始化你的参数。这个函数与之前的`initialize_parameters_random(...)`类似。唯一的区别是，你不是将 `np.random.randn(..,..)` 乘以 10，而是乘以 $\sqrt{\frac{2}{\text{dimension of the previous layer}}}$，这是 He 初始化建议使用 ReLU 激活值的层。 

In [ ]:
# GRADED FUNCTION: initialize_parameters_he

def initialize_parameters_he(layers_dims):
    """
    参数：
    layer_dims -- 包含每层大小的 python 数组（列表）。
    
    返回：
    parameters -- 包含参数 "W1"、"b1"、...、"WL"、"bL" 的 python 字典：
                    W1 -- 形状权重矩阵(layers_dims[1], layers_dims[0])
                    b1 -- 形状的偏置向量 (layers_dims[1], 1)
                    ...
                    WL -- 形状权重矩阵(layers_dims[L], layers_dims[L-1])
                    bL -- 形状的偏置向量 (layers_dims[L], 1)
    """
    
    np.random.seed(3)
    parameters = {}
    L = len(layers_dims) - 1 # 表示层数的整数
     
    for l in range(1, L + 1):
        #（约2行代码）
        # parameters['W' + str(l)] = 
        # parameters['b' + str(l)] =
        # YOUR CODE STARTS HERE
        
        
        # YOUR CODE ENDS HERE
        
    return parameters

In [ ]:
parameters = initialize_parameters_he([2, 4, 1])
print("W1 = " + str(parameters["W1"]))
print("b1 = " + str(parameters["b1"]))
print("W2 = " + str(parameters["W2"]))
print("b2 = " + str(parameters["b2"]))

initialize_parameters_he_test(initialize_parameters_he)
# 参数

**预期输出**

```
W1 = [[ 1.78862847  0.43650985]
 [ 0.09649747 -1.8634927 ]
 [-0.2773882  -0.35475898]
 [-0.08274148 -0.62700068]]
b1 = [[0.] [0.] [0.] [0.]]
W2 = [[-0.03098412 -0.33744411 -0.92904268  0.62552248]]
b2 = [[0.]]
```

运行以下代码，使用 He 初始化来训练你的模型 15,000 次迭代。

In [ ]:
parameters = model(train_X, train_Y, initialization = "he")
print ("On the train set:")
predictions_train = predict(train_X, train_Y, parameters)
print ("On the test set:")
predictions_test = predict(test_X, test_Y, parameters)

In [ ]:
plt.title("Model with He initialization")
axes = plt.gca()
axes.set_xlim([-1.5,1.5])
axes.set_ylim([-1.5,1.5])
plot_decision_boundary(lambda x: predict_dec(parameters, x.T), train_X, train_Y)

**观察**：
- 使用 He 初始化的模型在少量迭代中很好地分离了蓝点和红点。


<a name='7'></a>
## 7 - 结论

你已经尝试了三种不同类型的初始化。对于相同的迭代次数和相同的超参数，比较为：

<table> 
    <tr>
        <td>
            <b>模型</b>
        </td>
        <td>
            <b>训练精度</b>
        </td>
        <td>
            <b>问题/评论</b>
        </td>
    </tr>
        <td>
        带零初始化的 3 层神经网络
        </td>
        <td>
        50%
        </td>
        <td>
        未能打破对称性
        </td>
    <tr>
        <td>
        具有大随机初始化的 3 层神经网络
        </td>
        <td>
        83%
        </td>
        <td>
        权重太大 
        </td>
    </tr>
    <tr>
        <td>
        带 He 初始化的 3 层神经网络
        </td>
        <td>
        99%
        </td>
        <td>
        推荐方法
        </td>
    </tr>
</table> 

**恭喜**！你已完成初始化 Notebook。 

以下是主要内容的快速回顾：

<div style="color:blue;">

- 不同的初始化会导致截然不同的结果
- 随机初始化用于打破对称性并确保不同的隐藏单元可以学习不同的东西
- 拒绝初始化太大的值！
- 他的初始化对于 ReLU 激活值的网络效果很好

</div>